# Lecture 3.7 — Parallel Tool Calls and Tool Execution Order

**Course:** OpenAI Agents SDK — Complete Course  
**Section:** 03 — Tools: Extending Agent Capabilities

This notebook covers two controls for tool execution that are easy to mix up.

1. `ModelSettings.parallel_tool_calls`. This lives on the model side. It decides whether the model is allowed to send more than one tool call in a single turn.
2. `ToolExecutionConfig.max_function_tool_concurrency`. This lives on the SDK side. It decides how many of the calls the model already sent are actually run at the same time.

These two settings do not affect each other. In this notebook you will build three simple async tools, run four scenarios that combine the two settings in different ways, and observe exactly how each tool call was scheduled using a small timing helper. You will also see how the SDK orders tool results when they finish at different times, and what happens when several tools write to shared context at once.

A quick note before you start: total wall-clock timings around `Runner.run()` include real model latency, which varies between runs. This notebook prints tool start offsets alongside those totals specifically so the scheduling behaviour stays visible regardless of that variance.

## 1. Install the OpenAI Agents SDK

This cell installs the `openai-agents` package. The version is pinned so that every example in this notebook behaves exactly as shown here.

If you want the latest version instead, remove the version pin and run `pip install openai-agents`. You can also substitute any other version number you prefer.

If the package is already installed in your current session, pip will simply confirm that and move on.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.17.8 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.6/859.6 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00


## 2. Configure the OpenAI API Key

This notebook reads your OpenAI API key from Google Colab Secrets. Keeping the key in Secrets means it never sits inside the notebook file itself.

### How to add a secret in Google Colab

1. Click the key icon in the left sidebar, or go to **Tools → Secrets**.
2. Click **+ Add new secret**.
3. Set **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key into the **Value** field.
5. Turn **Notebook access** on.
6. Close the panel. The key is now available to this notebook.

### Running locally?

Set the environment variable in your terminal before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
```

Then you can skip the Colab-specific code below. The key will already be in `os.environ`.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 3. Set the Model Name

We declare a single `MODEL_NAME` variable here and use it everywhere an `Agent` is defined in this notebook. Changing this one variable updates the model used across every agent below.

The default is `gpt-5.4-mini`. Check the linked page for the latest available models.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## 4. Imports

Here is what we import and why.

| Import | Source | Purpose |
|---|---|---|
| `asyncio` | stdlib | `asyncio.sleep()` gives our tools simulated latency |
| `time` | stdlib | `time.time()` measures how long each scenario takes |
| `dataclass`, `field` | stdlib `dataclasses` | Define the `TaskTracker` context object used later |
| `Reasoning` | `openai.types.shared` | Turns off chain-of-thought so our timings stay clean |
| `Agent` | `agents` | The core agent class |
| `ModelSettings` | `agents` | Carries provider-side settings, including `parallel_tool_calls` |
| `RunConfig` | `agents` | Carries SDK-side run settings, including `tool_execution` |
| `RunContextWrapper` | `agents` | Type hint for the context object passed into tools |
| `Runner` | `agents` | Runs the agent loop through `await Runner.run()` |
| `ToolExecutionConfig` | `agents` | Dataclass with `max_function_tool_concurrency`. This is the SDK-side control |
| `function_tool` | `agents` | Decorator that turns a Python function into an agent tool |

`ToolExecutionConfig` is exported from `agents` at the top level, so it can be imported directly alongside everything else.

In [4]:
import asyncio
import time
from dataclasses import dataclass, field

from openai.types.shared import Reasoning

from agents import (
    Agent,
    ModelSettings,
    RunConfig,
    RunContextWrapper,
    Runner,
    ToolExecutionConfig,
    function_tool,
)

## 5. The Two Controls

Before running any code, let's lay out the two controls side by side. They sit at different layers, and once you see that clearly, the rest of the lecture is easy to follow.

### Control 1. `ModelSettings.parallel_tool_calls` (provider side)

This setting is sent to the LLM provider as part of the request. It decides whether the model is allowed to emit more than one tool call in a single response turn.

| Value | Behaviour |
|---|---|
| `None` (default) | Provider default, typically `True` |
| `True` | Model may emit multiple tool calls in one turn |
| `False` | Model emits exactly one tool call per turn |

### Control 2. `ToolExecutionConfig.max_function_tool_concurrency` (SDK side)

This setting never touches the provider. By the time it applies, the model has already emitted its tool calls. It decides how many of those calls the SDK runs at the same time.

| Value | Behaviour |
|---|---|
| `None` (default) | SDK starts every emitted call concurrently |
| `N` (integer, at least 1) | SDK runs at most `N` calls at the same time |

If you set a value below 1, the SDK raises a `ValueError` immediately.

### Why they are separate

From the SDK documentation: `parallel_tool_calls` controls whether the model is allowed to emit multiple tool calls in a single response, while `tool_execution.max_function_tool_concurrency` controls how the SDK executes local function tool calls after the model has emitted them.

In other words, one setting shapes what the model sends. The other shapes what the SDK does with what it received. You can mix and match them freely, and the four scenarios below walk through the combinations that matter most in practice.

## 6. Define Three Async Tools with Simulated Latency

We define three async tools, each simulating a one-second network call, like a weather API, a population lookup, or a timezone service.

We also add a small logging helper here. Every tool call records its own start time into a shared `tool_call_log` list. `reset_tool_call_log()` clears that list before a scenario runs, and `print_tool_call_log()` prints when each tool call started, relative to the first one in that run.

This matters because the `Completed in Xs` timing you saw for earlier scenarios includes real network and model latency, which varies from run to run and can easily hide the effect we are trying to demonstrate. The tool start offsets do not depend on how long the model itself took to respond, so they stay reliable even when the total wall-clock time jumps around between runs.

- If all three tools start at roughly the same offset, they ran concurrently.
- If tools start in a staircase pattern about one second apart, they ran one at a time.

This makes every scenario below directly and reliably observable, independent of API jitter.

In [5]:
tool_call_log: list[tuple[str, float]] = []


def reset_tool_call_log() -> None:
    """Clears the shared log before each scenario so timings don't mix runs together."""
    tool_call_log.clear()


def print_tool_call_log() -> None:
    """Prints when each tool call started, relative to the first call in this run.

    This reflects how the calls were actually scheduled and does not depend on
    model response latency, so it stays reliable even when total wall-clock time
    varies between runs.
    """
    if not tool_call_log:
        print("No tool calls were logged.")
        return
    first_start = min(start for _, start in tool_call_log)
    print("Tool start times (relative to the first tool call):")
    for name, start in sorted(tool_call_log, key=lambda entry: entry[1]):
        print(f"  {name}: +{start - first_start:.2f}s")


@function_tool
async def fetch_weather(city: str) -> str:
    """Fetches current weather for a city.

    Args:
        city: The city to fetch weather for.
    """
    tool_call_log.append(("fetch_weather", time.time()))
    await asyncio.sleep(1.0)
    return f"Weather in {city}: Sunny, 24\u00b0C."


@function_tool
async def fetch_population(city: str) -> str:
    """Fetches the population of a city.

    Args:
        city: The city to fetch population for.
    """
    tool_call_log.append(("fetch_population", time.time()))
    await asyncio.sleep(1.0)
    return f"Population of {city}: 2.1 million."


@function_tool
async def fetch_timezone(city: str) -> str:
    """Fetches the timezone of a city.

    Args:
        city: The city to fetch timezone for.
    """
    tool_call_log.append(("fetch_timezone", time.time()))
    await asyncio.sleep(1.0)
    return f"Timezone of {city}: UTC+5:30."

## 7. Scenario 1. Default Settings

This is the default behaviour. We do not touch either control.

- `parallel_tool_calls=None`, so the provider default applies, and the model is free to emit all three tool calls in one turn.
- `max_function_tool_concurrency=None`, so the SDK starts all emitted calls at the same time.
- All three one-second tools run in parallel, so the tool start offsets below should all read close to `+0.00s`.

A note on the `Completed in Xs` line: it measures the entire `Runner.run()` call, which includes two round trips to the model (one to decide the tool calls, one to write the final answer) on top of the tool execution itself. Model latency varies between runs, so this total will vary too, sometimes by several seconds. The tool start offsets printed below are the reliable signal to watch, since they reflect only how the tools were scheduled, not how long the model took to respond.

In [6]:
agent = Agent(
    name="City Info Agent",
    instructions=(
        "You are a city information assistant. "
        "When asked about a city, call ALL relevant tools "
        "to gather complete information."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        # parallel_tool_calls=None means the provider default applies (True)
    ),
    tools=[fetch_weather, fetch_population, fetch_timezone],
)

reset_tool_call_log()
start = time.time()
result = await Runner.run(
    agent,
    "Tell me everything about Mumbai: weather, "
    "population, and timezone.",
)
elapsed = time.time() - start

print(result.final_output)
print(f"\nCompleted in {elapsed:.1f}s (total wall-clock, includes model latency)")
print()
print_tool_call_log()

Mumbai:
- Weather: Sunny, 24°C
- Population: 2.1 million
- Timezone: UTC+5:30

Completed in 7.0s (total wall-clock, includes model latency)

Tool start times (relative to the first tool call):
  fetch_weather: +0.00s
  fetch_population: +0.00s
  fetch_timezone: +0.00s


## 8. Scenario 2. `parallel_tool_calls=False`

Now we tell the provider that the model may only emit one tool call per turn.

- The model calls one tool, waits for the result, then decides the next call.
- Three tools means three separate turns with the model, each carrying its own one-second tool plus a full model round trip.
- Compared to Scenario 1's two round trips (one to emit all three calls, one for the final answer), this scenario makes four round trips in total. That is where the extra time comes from, not from the tools themselves.

Because of that, do not expect the `Completed in Xs` totals to cleanly separate Scenario 1 and Scenario 2. Model latency dominates both numbers and varies between runs, so a single run can easily show similar or even reversed totals. What will not lie to you is the tool start offset pattern below: in this scenario, each tool call starts noticeably more than a second after the previous one, because the model has to be asked again before the next call happens. Compare that pattern to Scenario 1, where every offset is close to zero.

Reach for this setting when a later tool call genuinely depends on an earlier tool's result, or when the model needs to see one result before it can decide what to do next.

In [7]:
agent_sequential = Agent(
    name="Sequential City Agent",
    instructions=(
        "You are a city information assistant. "
        "When asked about a city, gather complete information."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        parallel_tool_calls=False,
    ),
    tools=[fetch_weather, fetch_population, fetch_timezone],
)

reset_tool_call_log()
start = time.time()
result = await Runner.run(
    agent_sequential,
    "Tell me everything about Chennai: weather, "
    "population, and timezone.",
)
elapsed = time.time() - start

print(result.final_output)
print(f"\nCompleted in {elapsed:.1f}s (total wall-clock, includes model latency)")
print()
print_tool_call_log()

Chennai:
- Weather: Sunny, 24°C
- Population: 2.1 million
- Timezone: UTC+5:30

Completed in 8.6s (total wall-clock, includes model latency)

Tool start times (relative to the first tool call):
  fetch_weather: +0.00s
  fetch_population: +2.03s
  fetch_timezone: +4.67s


## 9. Scenario 3. `max_function_tool_concurrency=2`

This time we leave the model free to emit all three calls in one turn, but we cap SDK-side concurrency at two.

`ToolExecutionConfig` is a run-level setting. It is passed through `RunConfig` to `Runner.run()`, not to the `Agent` definition. That means the very same agent can be run with different concurrency limits depending on context.

- Model emits all three calls in one turn, same as Scenario 1.
- SDK starts two tools immediately, and starts the third once a slot frees up.
- Watch the tool start offsets: two tools should start close to `+0.00s`, and the third should start close to `+1.00s`, once the first slot frees up.

As before, the total wall-clock time also includes model latency and will vary between runs. The offset pattern is what reliably shows the throttling effect.

This is the setting to reach for when you are calling a rate-limited external API or managing a limited resource pool, such as database connections.

In [8]:
reset_tool_call_log()
start = time.time()
result = await Runner.run(
    agent,  # parallel_tool_calls=None (default True)
    "Tell me everything about Delhi: weather, "
    "population, and timezone.",
    run_config=RunConfig(
        tool_execution=ToolExecutionConfig(
            max_function_tool_concurrency=2,
        ),
    ),
)
elapsed = time.time() - start

print(result.final_output)
print(f"\nCompleted in {elapsed:.1f}s (total wall-clock, includes model latency)")
print()
print_tool_call_log()

Delhi:
- Weather: Sunny, 24°C
- Population: 2.1 million
- Timezone: UTC+5:30

Completed in 5.3s (total wall-clock, includes model latency)

Tool start times (relative to the first tool call):
  fetch_weather: +0.00s
  fetch_population: +0.00s
  fetch_timezone: +1.00s


## 10. Scenario 4. `max_function_tool_concurrency=1`

Now we push SDK-side concurrency down to one, so tools run strictly one after another, even though the model still emits all three calls in a single turn.

- Model emits all three calls in one turn, exactly as in Scenario 1 and Scenario 3.
- SDK executes them one at a time.
- Watch the tool start offsets: expect a clean staircase, roughly `+0.00s`, `+1.00s`, `+2.00s`, one second apart, since each tool only starts once the previous one has finished.

Compare that staircase to Scenario 2. Both scenarios end up executing tools one at a time, but the underlying mechanism is different, and the tool start offsets make the difference visible:

| Setting | Model turns | Tool start offset pattern |
|---|---|---|
| `parallel_tool_calls=False` | Three separate turns | Gaps of roughly one second **plus** a model round trip each time |
| `max_function_tool_concurrency=1` | One turn | Gaps of almost exactly one second, since there is no model round trip between tool calls |

`max_function_tool_concurrency=1` reaches the same sequential execution with far fewer round trips to the model, which is why its staircase is tighter and more consistent than Scenario 2's. Reach for it when your tools share mutable state that is not safe to touch from more than one place at once.

In [9]:
reset_tool_call_log()
start = time.time()
result = await Runner.run(
    agent,
    "Tell me everything about Kolkata: weather, "
    "population, and timezone.",
    run_config=RunConfig(
        tool_execution=ToolExecutionConfig(
            max_function_tool_concurrency=1,
        ),
    ),
)
elapsed = time.time() - start

print(result.final_output)
print(f"\nCompleted in {elapsed:.1f}s (total wall-clock, includes model latency)")
print()
print_tool_call_log()

Kolkata:
- Weather: Sunny, 24°C
- Population: 2.1 million
- Timezone: UTC+5:30

Completed in 5.8s (total wall-clock, includes model latency)

Tool start times (relative to the first tool call):
  fetch_weather: +0.00s
  fetch_population: +1.00s
  fetch_timezone: +2.00s


## 11. Tool Result Ordering: Call Order, Not Completion Order

When tools run concurrently, they can finish in a different order than they were called. A quick tool called second can finish before a slow tool called first.

The SDK always assembles the final list of tool results in the order the calls were made, not the order they finished. Internally, the SDK keeps track of each tool run in its original call order and builds the result list by walking through that original order, regardless of which task actually completed first. That means the model always sees tool results in the same sequence it issued the calls, which keeps its reasoning consistent.

For this to be visible, both calls need to land in the same model turn. We set `parallel_tool_calls=True` explicitly here (rather than leaving it at the default `None`) to make sure the model batches both calls together instead of spacing them across two turns. `tool_choice="required"` makes sure the model actually calls the tool rather than answering directly.

In the cell below, we define a tool with variable latency. When called with `name="fast"` it returns in 0.1 seconds. Any other name takes a full second. We then instruct the model to call the tool twice: first with `name="slow"`, then with `name="fast"`. We reuse the `tool_call_log` helper from Section 6 to show the order the calls actually started in, and we print `result.new_items` to show the order the results ended up in. Even though `fast` finishes first, its result should still appear after `slow`'s result in `result.new_items`, because `slow` was called first.

In [10]:
@function_tool
async def variable_latency_tool(name: str) -> str:
    """A tool with variable latency to demonstrate ordering.

    Args:
        name: Tool name. 'fast' runs in 0.1s, anything else in 1s.
    """
    tool_call_log.append((name, time.time()))
    delay = 0.1 if name == "fast" else 1.0
    await asyncio.sleep(delay)
    return f"Result from {name} (latency={delay}s)"


order_agent = Agent(
    name="Order Test Agent",
    instructions=(
        "Call variable_latency_tool twice: "
        "first with name='slow' then with name='fast'."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="required",
        parallel_tool_calls=True,
    ),
    tools=[variable_latency_tool],
)

reset_tool_call_log()
result = await Runner.run(
    order_agent,
    "Run both tools now.",
)

print("Call order (when each call actually started):")
print_tool_call_log()

print("\nResult order (what the model received back, in result.new_items):")
for item in result.new_items:
    print(
        f"  {type(item).__name__}: "
        f"{getattr(item, 'output', getattr(item, 'type', ''))}"
    )

Call order (when each call actually started):
Tool start times (relative to the first tool call):
  slow: +0.00s
  fast: +0.00s

Result order (what the model received back, in result.new_items):
  ToolCallItem: tool_call_item
  ToolCallItem: tool_call_item
  ToolCallOutputItem: Result from slow (latency=1.0s)
  ToolCallOutputItem: Result from fast (latency=0.1s)
  MessageOutputItem: message_output_item


## 12. Context Mutation in Concurrent Tools

Tool results come back in call order, but shared context works differently. When multiple tools write to the same context object concurrently, the writes land in completion order, not call order.

This is ordinary asyncio behaviour, not a bug and not something the SDK tries to control. The context object is just a plain Python object shared across every concurrent coroutine, and nothing serialises writes to it.

As with the previous cell, we set `parallel_tool_calls=True` explicitly so the model batches all three calls into a single turn. Without that, the model could space the calls across separate turns, and completion order would trivially match call order for the wrong reason.

In the cell below:

- We define a `TaskTracker` dataclass with a `completed` list.
- We define a `tracked_task` tool that logs its own start into `tool_call_log`, sleeps for a given duration, then appends its name to `completed` on the way out.
- We ask the agent to run three tasks with different durations: `task_a` at 0.5s, `task_b` at 0.1s, `task_c` at 0.3s.

The model is instructed to call them in the order `task_a`, `task_b`, `task_c`. We print both the call order (from `tool_call_log`) and the completion order (from `tracker.completed`) so you can see them differ directly. Since they all run concurrently, the shortest one finishes first. Expect the call order to read `task_a`, `task_b`, `task_c`, and the completion order to read `task_b`, then `task_c`, then `task_a`.

In [11]:
@dataclass
class TaskTracker:
    completed: list = field(default_factory=list)


@function_tool
async def tracked_task(
    ctx: RunContextWrapper[TaskTracker],
    task_name: str,
    duration: float,
) -> str:
    """Performs a timed task and records completion order.

    Args:
        task_name: Name of the task.
        duration: Duration in seconds.
    """
    tool_call_log.append((task_name, time.time()))
    await asyncio.sleep(duration)
    ctx.context.completed.append(task_name)
    return f"{task_name} done."


tracker = TaskTracker()

tracker_agent = Agent(
    name="Tracker Agent",
    instructions=(
        "Run three tasks: "
        "task_a (duration=0.5), "
        "task_b (duration=0.1), "
        "task_c (duration=0.3)."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        parallel_tool_calls=True,
    ),
    tools=[tracked_task],
)

reset_tool_call_log()
result = await Runner.run(
    tracker_agent,
    "Run all three tasks.",
    context=tracker,
)

print("Call order (when each task actually started):")
print_tool_call_log()

print("\nCompletion order (from the shared context object):", tracker.completed)
print("Final output:", result.final_output)

Call order (when each task actually started):
Tool start times (relative to the first tool call):
  task_a: +0.00s
  task_b: +0.00s
  task_c: +0.00s

Completion order (from the shared context object): ['task_b', 'task_c', 'task_a']
Final output: Done.


## 13. Summary

### A note on measuring this reliably

The `Completed in Xs` totals you saw in Scenarios 1 through 4 include real model round-trip latency, which varies from run to run and can be large enough to hide the effect we are demonstrating. If you compare two scenarios and the totals look similar or even reversed, that is expected, not a bug. The tool start offsets printed by `print_tool_call_log()` are the reliable signal, since they reflect only how the tools were scheduled. If you want to see the pattern more clearly, try re-running a couple of the scenario cells two or three times and comparing the offset patterns rather than the single totals.

### Timing reference (tool-scheduling behaviour, not wall-clock totals)

For three independent tools that each take one second:

| `parallel_tool_calls` | `max_function_tool_concurrency` | Behaviour | Tool start offset pattern |
|---|---|---|---|
| `None` / `True` (default) | `None` (default) | All emitted in one turn, all run concurrently | All offsets close to `+0.00s` |
| `None` / `True` (default) | `2` | All emitted in one turn, run two at a time | Two near `+0.00s`, one near `+1.00s` |
| `None` / `True` (default) | `1` | All emitted in one turn, run one at a time | Staircase, about `+0.00s`, `+1.00s`, `+2.00s` |
| `False` | `None` | One emitted per turn, executed as emitted | Staircase, but each step also includes a model round trip |

### Decision guide

| Situation | Recommended setting |
|---|---|
| Independent, fast, I/O-bound tools | Leave both settings at their defaults |
| Calling a rate-limited external API | `max_function_tool_concurrency=N` |
| One tool depends on another's result | `parallel_tool_calls=False` |
| Tools share mutable state that is not concurrency-safe | `max_function_tool_concurrency=1` |

### Key facts to remember

- Tool results always return to the model in call order, not completion order. The SDK handles this for you.
- Shared context mutation follows completion order under concurrent execution. Keep this in mind whenever you accumulate state across concurrent tool calls.
- `ToolExecutionConfig` is a run-level setting passed through `RunConfig`, not an agent-level setting.
- Total wall-clock time around `Runner.run()` includes model latency and will vary between runs. Tool start offsets are the more reliable way to observe scheduling behaviour.
- Always use `await Runner.run()` in Jupyter or Colab. `Runner.run_sync()` raises a `RuntimeError` in environments that already have an event loop running.